
# Notebook 4 - QLoRA Fine-Tuning (Colab + uv)

This notebook fine-tunes the model used in your agentic address intelligence system.

Locked architecture stays the same:
1. Coordinator Agent
2. Retrieval Agent
3. Address Standardization Agent
4. Commute & Confidence Agent
5. Eligibility Agent
6. Escalation Agent

Notebook 4 focuses on the fine-tuning part.

What this notebook does:
- uses `uv` for package management
- optionally uploads helper files if you want them in Colab
- reads employee + KB data from BigQuery
- creates a synthetic supervised fine-tuning dataset
- formats the data for instruction tuning
- loads a base Hugging Face model in 4-bit
- attaches LoRA adapters with PEFT
- configures QLoRA training
- logs metrics to Weights & Biases
- trains and evaluates
- saves the LoRA adapter for later notebooks



## Step 0: install uv


In [ ]:

!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] += ":/root/.local/bin"
!uv --version


In [ ]:
!pip uninstall -y numpy pandas faiss-cpu sentence-transformers transformers accelerate bitsandbytes langchain langchain-core langchain-community langchain-huggingface
!pip uninstall -y trl transformers peft accelerate bitsandbytes
!pip install --no-cache-dir \
  numpy==1.26.4 \
  pandas==2.2.2 \
  langchain \
  langchain-community \
  langchain-huggingface \
  faiss-cpu \
  sentence-transformers \
  transformers \
  accelerate \
  bitsandbytes \
  google-cloud-bigquery \
  db-dtypes \
  requests==2.32.4

In [ ]:
!pip uninstall -y trl transformers peft accelerate bitsandbytes
!pip install -U "trl[peft]" bitsandbytes datasets wandb

In [ ]:
import transformers, trl, peft, accelerate
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)

from peft import get_peft_model, LoraConfig, prepare_model_for_kbit_training
print("PEFT import OK")


## Step 1: install project packages with uv


In [ ]:

import os
os.environ["PATH"] += ":/root/.local/bin"

#!uv pip install --system   numpy==1.26.4   pandas   datasets   transformers   accelerate   bitsandbytes   peft   trl   wandb   google-cloud-bigquery   db-dtypes   requests==2.32.4


In [ ]:
import numpy as np
import pandas as pd
import torch

print(np.__version__)
print(pd.__version__)
print(torch.cuda.is_available())




## Step 2: optional helper file upload

Optional files:
- `multi_agent_helpers_uv.py`
- `retrieval_agent_final_helpers_uv.py`
- `finetuning_helpers_uv.py`

This notebook is self-contained, so upload is optional.


In [ ]:

from google.colab import files
import os

print("Current files before optional upload:")
print(os.listdir())

# OPTIONAL:
# uploaded = files.upload()



## Step 3: imports


In [ ]:

import os
import json
import random
import torch
import pandas as pd
import wandb

from google.colab import auth
from google.cloud import bigquery

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline
)
from peft import LoraConfig, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig



## Step 4: authenticate and connect to BigQuery


In [ ]:

auth.authenticate_user()
print("Authenticated successfully")

PROJECT_ID = "" #add your project id here
DATASET_ID = "address_intelligence_demo" #add your database id here
TABLE_EMPLOYEES = "employee_input"
TABLE_KB = "kb_documents"

client = bigquery.Client(project=PROJECT_ID)
print("Connected to BigQuery project:", PROJECT_ID)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



## Step 5: read employee + KB data from BigQuery


In [ ]:

kb_query = f'''
SELECT doc_id, source_type, title, text
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_KB}`
ORDER BY doc_id
'''

employee_query = f'''
SELECT employee_id, employee_name, office_id, raw_home_address, office_address
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_EMPLOYEES}`
ORDER BY employee_id
'''

kb_df = client.query(kb_query).to_dataframe()
employee_df = client.query(employee_query).to_dataframe()

print("KB rows:", len(kb_df))
print("Employee rows:", len(employee_df))
employee_df.head()



## Step 6: build a synthetic supervised fine-tuning dataset

We fine-tune mainly for the Address Standardization Agent.


In [ ]:

seed_examples = [
    {
        "raw_home_address": "1818 Church Street, Nashvile, TN 37203",
        "office_address": "100 Healthcare Blvd, Nashville, TN 37205",
        "corrected_standardized_address": "1818 Church St, Nashville, TN 37203",
        "correction_confidence": 0.95,
        "standardization_reason": "Corrected the city typo and standardized the street suffix using KB typo examples."
    },
    {
        "raw_home_address": "99 broadwai, nashville, tn 37201",
        "office_address": "100 Healthcare Blvd, Nashville, TN 37205",
        "corrected_standardized_address": "99 Broadway, Nashville, TN 37201",
        "correction_confidence": 0.94,
        "standardization_reason": "Corrected the misspelled street name and normalized capitalization and state abbreviation."
    },
    {
        "raw_home_address": "123 Main Street Apartment Two Nashville Tennessee 37203",
        "office_address": "100 Healthcare Blvd, Nashville, TN 37205",
        "corrected_standardized_address": "123 Main St Apt 2, Nashville, TN 37203",
        "correction_confidence": 0.90,
        "standardization_reason": "Normalized street suffix, apartment format, punctuation, and state abbreviation."
    },
    {
        "raw_home_address": "12 Music Sq W Nashville TN",
        "office_address": "100 Healthcare Blvd, Nashville, TN 37205",
        "corrected_standardized_address": "12 Music Sq W, Nashville, TN",
        "correction_confidence": 0.68,
        "standardization_reason": "Standardized punctuation and formatting, but ZIP code remains missing so confidence is lower."
    },
    {
        "raw_home_address": "404 Not Found Ln, Brentwood, TN 37027",
        "office_address": "100 Healthcare Blvd, Nashville, TN 37205",
        "corrected_standardized_address": "404 Not Found Ln, Brentwood, TN 37027",
        "correction_confidence": 0.40,
        "standardization_reason": "Address appears suspicious, so it was preserved without inventing unsupported changes."
    }
]

employee_based_examples = []
for _, row in employee_df.iterrows():
    raw = row["raw_home_address"]
    corrected = raw

    corrected = corrected.replace("Broudway", "Broadway")
    corrected = corrected.replace("broadwai", "Broadway")
    corrected = corrected.replace("Nashvile", "Nashville")
    corrected = corrected.replace("Murfresboro", "Murfreesboro")
    corrected = corrected.replace("Peachtre", "Peachtree")
    corrected = corrected.replace("Apartment Two", "Apt 2")
    corrected = corrected.replace("Street", "St")
    corrected = corrected.replace("Tennessee", "TN")
    corrected = " ".join(corrected.split())

    confidence = 0.88
    reason = "Standardized the address using known typo-correction and formatting patterns."

    lower_raw = raw.lower()
    if "cupertino" in lower_raw or "new york" in lower_raw or "atlanta" in lower_raw:
        confidence = 0.92
        reason = "Address already appears structurally valid and required minimal standardization."
    if "music sq" in lower_raw and "372" not in lower_raw:
        confidence = 0.70
        reason = "Formatting was standardized, but ZIP code remains missing."
    if "404 not found" in lower_raw:
        confidence = 0.42
        reason = "Address appears suspicious, so no unsupported correction was invented."

    employee_based_examples.append({
        "raw_home_address": raw,
        "office_address": row["office_address"],
        "corrected_standardized_address": corrected,
        "correction_confidence": round(confidence, 2),
        "standardization_reason": reason
    })

sft_examples = seed_examples + employee_based_examples
print("Total synthetic SFT examples:", len(sft_examples))
pd.DataFrame(sft_examples).head(10)



## Step 7: format examples into instruction-tuning samples


In [ ]:

SYSTEM_PROMPT_ADDRESS_AGENT = """
You are an address standardization agent for an enterprise address intelligence system.

Use the retrieved policy, typo examples, and standardization rules to correct and standardize the employee address.

Rules:
1. Preserve valid information.
2. Correct obvious typos if strongly supported by evidence.
3. Do not invent unsupported address fields.
4. If an important field is missing, keep it missing and lower confidence.
5. Return valid JSON only.

Required JSON keys:
corrected_standardized_address
correction_confidence
standardization_reason
"""

retrieved_context_stub = [
    {
        "doc_id": row["doc_id"],
        "title": row["title"],
        "source_type": row["source_type"],
        "text": row["text"]
    }
    for _, row in kb_df.head(4).iterrows()
]

def make_chat_example(example):
    user_payload = {
        "raw_home_address": example["raw_home_address"],
        "office_address": example["office_address"],
        "retrieved_context": retrieved_context_stub
    }

    assistant_payload = {
        "corrected_standardized_address": example["corrected_standardized_address"],
        "correction_confidence": example["correction_confidence"],
        "standardization_reason": example["standardization_reason"]
    }

    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT_ADDRESS_AGENT},
            {"role": "user", "content": json.dumps(user_payload, indent=2)},
            {"role": "assistant", "content": json.dumps(assistant_payload, indent=2)}
        ]
    }

chat_examples = [make_chat_example(x) for x in sft_examples]
chat_examples[0]



## Step 8: train / eval split


In [ ]:

random.seed(42)
random.shuffle(chat_examples)

split_idx = max(1, int(len(chat_examples) * 0.8))
train_examples = chat_examples[:split_idx]
eval_examples = chat_examples[split_idx:]

print("Train size:", len(train_examples))
print("Eval size:", len(eval_examples))


In [ ]:

train_dataset = Dataset.from_list(train_examples)
eval_dataset = Dataset.from_list(eval_examples)

train_dataset[0]


In [ ]:
import transformers, trl, peft, accelerate
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)

In [ ]:

WANDB_PROJECT = "address-intelligence-qlora"
WANDB_RUN_NAME = "notebook4-qlora-address-agent"
BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        "base_model": BASE_MODEL_NAME,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        "epochs": 3,
        "per_device_train_batch_size": 2,
        "per_device_eval_batch_size": 2,
        "gradient_accumulation_steps": 4,
        "learning_rate": 2e-4,
        "optimizer": "adamw_torch",
        "bf16": bool(torch.cuda.is_available()),
        "quantization": "4bit_nf4",
    }
)

print("W&B initialized")
#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

In [ ]:


EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

VECTOR_DIR = "rag_vectorstore_final"
vectorstore.save_local(VECTOR_DIR)

print("Embedding model ready:", EMBEDDING_MODEL_NAME)
print("FAISS vector store created")
print("Saved vector store to:", VECTOR_DIR)


BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
model_name = BASE_MODEL_NAME   # your model name
WANDB_PROJECT = "address-intelligence-qlora"
WANDB_RUN_NAME = "notebook4-qlora-address-agent"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model.config.use_cache = False
base_model = prepare_model_for_kbit_training(base_model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

# Wrap manually ONCE
model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

training_args = SFTConfig(
    output_dir="address_agent_qlora_adapter",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    optim="adamw_torch",
    bf16=torch.cuda.is_available(),
    logging_steps=1,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    report_to="wandb",
    run_name=WANDB_RUN_NAME,
    save_total_limit=2,
    load_best_model_at_end=False,
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
)

trainer = SFTTrainer(
    model=model,                     # already wrapped
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("Trainer ready")


## Step 15: train the QLoRA adapter


In [ ]:
# 1) Train FIRST
train_result = trainer.train()

# 2) Then evaluate
# eval_metrics = trainer.evaluate()
# print(eval_metrics)


## Step 16: evaluate


In [ ]:
eval_logs = [x for x in trainer.state.log_history if "eval_loss" in x]
eval_logs


## Step 17: save the LoRA adapter + tokenizer


In [ ]:

OUTPUT_DIR = "address_agent_qlora_adapter"
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved adapter and tokenizer to:", OUTPUT_DIR)


In [ ]:
!zip -r address_agent_qlora_adapter.zip address_agent_qlora_adapter

In [ ]:
from google.colab import files
files.download("address_agent_qlora_adapter.zip")


## Step 18: quick inference test with the fine-tuned adapter


In [ ]:

inference_base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto",
    trust_remote_code=True
)

inference_model = PeftModel.from_pretrained(inference_base, OUTPUT_DIR)

inference_pipe = pipeline(
    "text-generation",
    model=inference_model,
    tokenizer=tokenizer,
    max_new_tokens=180,
    do_sample=False,
    return_full_text=True
)

sample_eval = sft_examples[0]
messages = [
    {"role": "system", "content": SYSTEM_PROMPT_ADDRESS_AGENT},
    {"role": "user", "content": json.dumps({
        "raw_home_address": sample_eval["raw_home_address"],
        "office_address": sample_eval["office_address"],
        "retrieved_context": retrieved_context_stub
    }, indent=2)}
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
generated = inference_pipe(prompt)[0]["generated_text"]
answer = generated[len(prompt):]

print("MODEL OUTPUT:")
print(answer)



## Step 19: save local artifacts


In [ ]:

with open("notebook4_sft_examples.json", "w") as f:
    json.dump(sft_examples, f, indent=2)

with open("notebook4_eval_metrics.json", "w") as f:
    json.dump(eval_metrics, f, indent=2)

artifact_summary = {
    "base_model": BASE_MODEL_NAME,
    "adapter_dir": OUTPUT_DIR,
    "train_size": len(train_examples),
    "eval_size": len(eval_examples)
}
with open("notebook4_artifact_summary.json", "w") as f:
    json.dump(artifact_summary, f, indent=2)

print("Saved files:")
print("- notebook4_sft_examples.json")
print("- notebook4_eval_metrics.json")
print("- notebook4_artifact_summary.json")



## What we finished

Notebook 4 now covers:
- QLoRA fine-tuning
- 4-bit NF4 quantization
- LoRA adapters with PEFT
- hyperparameters for training
- W&B tracking
- train / eval loss logging
- saved adapter for later notebooks

This notebook fine-tunes the model mainly for the Address Standardization Agent.
Later, Notebook 5 will evaluate results more deeply, and Notebook 6 will prepare deployment.


Creating helper file (optional)

In [ ]:


from google.colab import drive
drive.mount('/content/drive')

import os

project_path = "/content/drive/MyDrive/address-intelligence-project"
os.listdir("/content/drive/MyDrive/address-intelligence-project")
os.listdir("/content/drive/MyDrive/address-intelligence-project/src")

In [ ]:
%%writefile /content/drive/MyDrive/address-intelligence-project/src/finetune_helper.py
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training


def load_tokenizer(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def load_model_with_qlora(model_name):
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_quant_type="nf4",
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_config,
        device_map="auto",
        trust_remote_code=True,
    )

    model = prepare_model_for_kbit_training(model)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )

    model = get_peft_model(model, lora_config)
    return model